# 02 · Ingeniería de características

Este notebook documenta las variables que usa el pipeline para estimar el total del siguiente mes. Los cálculos de producción provienen de `data.py` y `features.py`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from superstore_forecasting.config import (
    EVAL_END,
    EVAL_START,
    RAW_PATH,
    TUNING_END,
    TUNING_START,
)
from superstore_forecasting.data import load_orders
from superstore_forecasting.features import FEATURES, aggregate_monthly, make_features

root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "src" / "superstore_forecasting").is_dir()
)
dataset_path = root / RAW_PATH
if not dataset_path.is_file():
    raise FileNotFoundError(
        "Desde la raíz, ejecuta: uv run --locked python -m scripts.download_data"
    )

plt.style.use("seaborn-v0_8-whitegrid")
orders = load_orders(dataset_path)
monthly = aggregate_monthly(orders)

## De transacciones al objetivo mensual

Para el mes objetivo **t**, el target es la suma de `Sales` de todas sus líneas. Por cada mes, `aggregate_monthly` también cuenta pedidos distintos y calcula el ticket promedio como ventas divididas por el conteo. Un mes ausente detiene el flujo en vez de añadir un cero sin evidencia.

In [ ]:
display(monthly.head())

## Variables usadas por Ridge

| Variable | Observación disponible al cierre anterior |
|---|---|
| `sales_lag_1` | Ventas de t−1 |
| `sales_lag_12` | Ventas de t−12 |
| `orders_lag_1` | Pedidos distintos de t−1 |
| `ticket_lag_1` | Ticket promedio de t−1 |

El ingenuo estacional predice directamente `sales_lag_12`. El mes calendario identifica la fila y sirve para desplazar observaciones y fijar los cortes. El modelo actual no añade medias móviles ni variables de calendario codificadas.

In [ ]:
features = make_features(monthly)
observed = features.join(monthly["sales"].rename("target_sales"))
usable = observed.dropna(subset=[*FEATURES, "target_sales"])
display(usable.head().round(2))
display(observed.iloc[:-1].isna().sum().rename("Meses observados sin valor").to_frame())
display(observed.tail(1).round(2))

Los primeros meses tienen faltantes esperados porque aún no existe el historial requerido. El ajuste usa filas con predictores y objetivo observados. La última fila representa el siguiente mes. Tiene predictores históricos, pero aún carece de target.

In [ ]:
target = pd.Period(EVAL_START, freq="M")
window = monthly.loc[target - 12 : target, "sales"]
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
window.set_axis(window.index.to_timestamp()).plot(ax=ax, color="#267b82", marker="o")
for offset, label, color in (
    (12, "t−12", "#4d65a8"),
    (1, "t−1", "#bf6335"),
    (0, "t: objetivo", "#822d42"),
):
    period = target - offset
    ax.scatter(
        period.to_timestamp(),
        window.at[period],
        color=color,
        s=65,
        label=label,
        zorder=5,
    )
ax.set(
    title=f"Origen de los rezagos para {target}",
    ylabel="Ventas (unidad de Sales)",
    xlabel="Mes observado",
)
ax.legend()
plt.show()
display(observed.loc[[target]].round(2))

## Fuga temporal y validación walk-forward

Para cada mes evaluado se vuelven a ajustar Ridge y su escalado con historia situada estrictamente antes de ese mes. Una vez observado el resultado, este puede entrar en el siguiente fold. Se selecciona alpha en 2017 y se fija para la comparación de 2018. El siguiente gráfico muestra folds de ejemplo y distingue el historial usado para generar rezagos de las filas completas que puede usar Ridge.

In [ ]:
months = pd.PeriodIndex(
    [TUNING_START, "2017-06", TUNING_END, EVAL_START, "2018-06", EVAL_END], freq="M"
)
timeline = monthly.index
first_fit = usable.index[0]
fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")
for row, month in enumerate(months):
    end = timeline.get_loc(month)
    begin = timeline.get_loc(first_fit)
    ax.barh(
        row,
        begin,
        height=0.55,
        color="#9db6bd",
        label="Historial para rezagos" if row == 0 else None,
    )
    ax.barh(
        row,
        end - begin,
        left=begin,
        height=0.55,
        color="#267b82",
        label="Filas para ajustar Ridge" if row == 0 else None,
    )
    ax.barh(
        row,
        1,
        left=end,
        height=0.55,
        color="#bf6335",
        label="Mes evaluado" if row == 0 else None,
    )
ticks = range(0, len(timeline), 6)
ax.set_xticks([i + 0.5 for i in ticks], [str(timeline[i]) for i in ticks], rotation=30)
ax.set_yticks(range(len(months)), [str(month) for month in months])
ax.invert_yaxis()
ax.set(
    title="Ventanas de entrenamiento creciente y evaluación de un mes",
    xlabel="Mes calendario",
    ylabel="Fold",
)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncols=3)
plt.show()

In [ ]:
cutoff = target - 1
from_history = make_features(monthly.loc[:cutoff]).loc[target, list(FEATURES)]
pd.testing.assert_series_equal(from_history, features.loc[target, list(FEATURES)])
display(
    Markdown(
        f"La fila de **{target}** coincide al reconstruirla solo con datos hasta **{cutoff}**."
    )
)

## Conclusiones

Los rezagos, el volumen de pedidos y el ticket previo se conocen al cierre anterior. La ventana inicial sin historial anual reduce el número de filas de entrenamiento. El corte operativo requiere confirmar que el último mes se ha cerrado antes de inferir.